# Standard Colab Backtest Runner

Runs a Strategy Contract v1 module without changing its trading logic. `RUN_MODE` selects a canonical trade log for one backtest or a trusted metrics summary for a parameter sweep. The notebook owns installation, execution, export, validation, and download.


In [ ]:
# Automatic setup. Run all cells; do not copy or retype this command.
%pip install --quiet "trade-log-exporter @ git+https://github.com/Divo15/Trade-log-Analytics-DashBoard.git@main"
# Add only packages required by the real supported engine.
%pip install --quiet pandas duckdb


## Upload strategy and engine files

Upload the generated strategy module and every local Python file required by its real engine. The strategy module declares `RUN_MODE` and, for a sweep, the exact `SWEEP_PARAMETER_SETS`.


In [ ]:
from google.colab import files

uploaded_code = files.upload()
print("Uploaded:", ", ".join(uploaded_code))


## Select market data

Choose `drive` and enter your selected Drive file/folder path, or choose `upload` and upload one data file or ZIP archive. The strategy receives only `context.market_data`.


In [ ]:
from pathlib import Path
import shutil
from google.colab import drive, files

DATA_SOURCE = "drive"  # "drive" or "upload"
MARKET_DATA_PATH = ""  # Required for drive; select your own file or folder.

if DATA_SOURCE == "drive":
    drive.mount("/content/drive")
    if not MARKET_DATA_PATH:
        raise ValueError("Set MARKET_DATA_PATH to your selected Google Drive file or folder")
    market_data = Path(MARKET_DATA_PATH)
elif DATA_SOURCE == "upload":
    uploaded_data = files.upload()
    if len(uploaded_data) != 1:
        raise ValueError("Upload exactly one market-data file or one ZIP archive")
    uploaded_path = Path(next(iter(uploaded_data)))
    if uploaded_path.suffix.lower() == ".zip":
        destination = Path("/content/market_data")
        destination.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(uploaded_path), str(destination))
        market_data = destination
    else:
        market_data = uploaded_path
else:
    raise ValueError("DATA_SOURCE must be 'drive' or 'upload'")

if not market_data.exists():
    raise FileNotFoundError(f"Selected market data does not exist: {market_data}")
print("Market data:", market_data)


## Configure shared run assumptions

Enter only user-selected data and execution assumptions. Strategy-specific values remain in the generated module for single mode or its declared sweep parameter sets.


In [ ]:
STRATEGY_MODULE = "strategy"  # Filename without .py

INSTRUMENT = {"symbol": "NIFTY", "timeframe": "5m"}
PERIOD = {"start_date": "2023-01-02", "end_date": "2026-05-05"}
EXECUTION = {
    "capital_per_trade": 300000.0,
    "lot_size": 50,
    "slippage": 0.005,
    "fees_per_leg": 0.0,
    "timezone": "Asia/Kolkata",
}
SINGLE_PARAMETERS = {
    # Used only when RUN_MODE == "single". Add user-intended overrides here.
}


In [ ]:
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from types import MappingProxyType
from typing import Any
from uuid import uuid4
import importlib

def freeze(value):
    if isinstance(value, dict):
        return MappingProxyType({key: freeze(item) for key, item in value.items()})
    if isinstance(value, list):
        return tuple(freeze(item) for item in value)
    return value

@dataclass(frozen=True)
class StrategyContext:
    run_id: str
    market_data: Any
    config: Mapping[str, Any]

def make_context(run_id, parameters):
    return StrategyContext(
        run_id=run_id,
        market_data=market_data,
        config=freeze({
            "instrument": INSTRUMENT,
            "period": PERIOD,
            "execution": EXECUTION,
            "parameters": dict(parameters),
        }),
    )

importlib.invalidate_caches()
strategy = importlib.import_module(STRATEGY_MODULE)
if getattr(strategy, "STRATEGY_CONTRACT_VERSION", None) != "1":
    raise RuntimeError("Strategy must declare STRATEGY_CONTRACT_VERSION = '1'")
run_mode = getattr(strategy, "RUN_MODE", None)
if run_mode not in {"single", "sweep"}:
    raise RuntimeError("Strategy must declare RUN_MODE = 'single' or 'sweep'")
sweep_parameter_sets = getattr(strategy, "SWEEP_PARAMETER_SETS", None)
if run_mode == "single" and sweep_parameter_sets not in ((), []):
    raise RuntimeError("Single mode requires SWEEP_PARAMETER_SETS = ()")
if run_mode == "sweep":
    if not isinstance(sweep_parameter_sets, Sequence) or not sweep_parameter_sets:
        raise RuntimeError("Sweep mode requires non-empty SWEEP_PARAMETER_SETS")
    if not all(isinstance(item, Mapping) for item in sweep_parameter_sets):
        raise RuntimeError("Every sweep parameter set must be a mapping")
if not callable(getattr(strategy, "run_strategy", None)):
    raise RuntimeError("Strategy must expose callable run_strategy(context)")
print("Run mode:", run_mode)


## Execute and create the correct artifact

`single` produces `trades.csv`. `sweep` runs every declared parameter mapping and produces `sweep_results.csv` with trusted P&L, yearly P&L, drawdown, win rate, maximum loss, and related metrics calculated from validated canonical trades.


In [ ]:
from pathlib import Path
from trade_log_exporter import (
    export_sweep_summary,
    export_trade_log,
    validate_sweep_summary_csv,
    validate_trade_log_csv,
)

if run_mode == "single":
    context = make_context(f"single-{uuid4().hex}", SINGLE_PARAMETERS)
    result = strategy.run_strategy(context)
    output_path = Path("/content/output/trades.csv")
    receipt = export_trade_log(
        result["completed_trades"],
        output_path,
        mapper=result.get("trade_mapper"),
        expected_count=int(result["completed_trade_count"]),
    )
    validation = validate_trade_log_csv(receipt.output_path)
    if validation.row_count != int(result["completed_trade_count"]):
        raise RuntimeError("Validated row count differs from authoritative engine count")
    report = {
        "run_mode": run_mode,
        "status": "no_trades" if receipt.row_count == 0 else "succeeded",
        "completed_trade_count": int(result["completed_trade_count"]),
        "exported_row_count": receipt.row_count,
        "schema_version": receipt.schema_version,
        "validation": "passed",
    }
else:
    sweep_id = f"sweep-{uuid4().hex}"
    iterations = []
    for index, parameters in enumerate(sweep_parameter_sets):
        run_id = f"{sweep_id}-{index:06d}"
        result = strategy.run_strategy(make_context(run_id, parameters))
        iterations.append({
            "run_id": run_id,
            "parameters": dict(parameters),
            "result": result,
        })
    output_path = Path("/content/output/sweep_results.csv")
    receipt = export_sweep_summary(iterations, output_path, sweep_id=sweep_id)
    validation = validate_sweep_summary_csv(receipt.output_path)
    if validation.row_count != len(sweep_parameter_sets):
        raise RuntimeError("Validated row count differs from requested sweep iterations")
    report = {
        "run_mode": run_mode,
        "status": "succeeded",
        "sweep_id": sweep_id,
        "iteration_count": len(sweep_parameter_sets),
        "exported_row_count": receipt.row_count,
        "schema_version": receipt.schema_version,
        "validation": "passed",
    }

print(report)


## Download artifacts

Upload `trades.csv` as a single backtest or `sweep_results.csv` as a sweep. Keep the checksum manifest with the run.


In [ ]:
from google.colab import files

files.download(str(receipt.output_path))
files.download(str(receipt.manifest_path))
